In [5]:
import tensorflow as tf
import numpy as np

text = open("/content/sample_data/generated_text_10000_words.txt").read().lower()
print(f"Length: {len(text)}")

Length: 43915


In [6]:
# Encoding the text char -> index -> int

char = sorted(set(text))
print(char)
charidx = {ch: idx for idx, ch in enumerate(char)}
print(charidx)
print(charidx.items())
idxchar = {idx: ch for ch, idx in charidx.items()}
print(idxchar)
vocab_size = len(char)
encoded = [charidx[ch] for ch in text]
print(encoded)

[' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y']
{' ': 0, 'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'k': 10, 'l': 11, 'm': 12, 'n': 13, 'o': 14, 'p': 15, 'r': 16, 's': 17, 't': 18, 'u': 19, 'v': 20, 'w': 21, 'y': 22}
dict_items([(' ', 0), ('a', 1), ('b', 2), ('c', 3), ('d', 4), ('e', 5), ('f', 6), ('g', 7), ('h', 8), ('i', 9), ('k', 10), ('l', 11), ('m', 12), ('n', 13), ('o', 14), ('p', 15), ('r', 16), ('s', 17), ('t', 18), ('u', 19), ('v', 20), ('w', 21), ('y', 22)])
{0: ' ', 1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'k', 11: 'l', 12: 'm', 13: 'n', 14: 'o', 15: 'p', 16: 'r', 17: 's', 18: 't', 19: 'u', 20: 'v', 21: 'w', 22: 'y'}
[18, 8, 5, 12, 0, 1, 0, 21, 9, 18, 8, 0, 1, 16, 5, 0, 14, 19, 18, 0, 17, 5, 5, 0, 9, 0, 13, 5, 21, 0, 2, 5, 0, 4, 9, 4, 0, 13, 14, 18, 0, 12, 5, 0, 18, 8, 5, 9, 16, 0, 22, 14, 19, 0, 18, 8, 5, 16, 5, 0, 12, 5, 0, 14, 13, 5, 0, 12, 1

In [7]:
# Create Input Output Pair

seq_length = 5
sequences = []
labels = []

for i in range(len(encoded) - seq_length):
  a = i + seq_length
  seq = encoded[i:a]
  label = encoded[a]
  sequences.append(seq)
  labels.append(label)

print(sequences)
print(labels)

[[18, 8, 5, 12, 0], [8, 5, 12, 0, 1], [5, 12, 0, 1, 0], [12, 0, 1, 0, 21], [0, 1, 0, 21, 9], [1, 0, 21, 9, 18], [0, 21, 9, 18, 8], [21, 9, 18, 8, 0], [9, 18, 8, 0, 1], [18, 8, 0, 1, 16], [8, 0, 1, 16, 5], [0, 1, 16, 5, 0], [1, 16, 5, 0, 14], [16, 5, 0, 14, 19], [5, 0, 14, 19, 18], [0, 14, 19, 18, 0], [14, 19, 18, 0, 17], [19, 18, 0, 17, 5], [18, 0, 17, 5, 5], [0, 17, 5, 5, 0], [17, 5, 5, 0, 9], [5, 5, 0, 9, 0], [5, 0, 9, 0, 13], [0, 9, 0, 13, 5], [9, 0, 13, 5, 21], [0, 13, 5, 21, 0], [13, 5, 21, 0, 2], [5, 21, 0, 2, 5], [21, 0, 2, 5, 0], [0, 2, 5, 0, 4], [2, 5, 0, 4, 9], [5, 0, 4, 9, 4], [0, 4, 9, 4, 0], [4, 9, 4, 0, 13], [9, 4, 0, 13, 14], [4, 0, 13, 14, 18], [0, 13, 14, 18, 0], [13, 14, 18, 0, 12], [14, 18, 0, 12, 5], [18, 0, 12, 5, 0], [0, 12, 5, 0, 18], [12, 5, 0, 18, 8], [5, 0, 18, 8, 5], [0, 18, 8, 5, 9], [18, 8, 5, 9, 16], [8, 5, 9, 16, 0], [5, 9, 16, 0, 22], [9, 16, 0, 22, 14], [16, 0, 22, 14, 19], [0, 22, 14, 19, 0], [22, 14, 19, 0, 18], [14, 19, 0, 18, 8], [19, 0, 18, 8, 5], 

In [8]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

padd_sequences = pad_sequences(sequences, maxlen = seq_length, padding = 'post')

# Convert to numpy arrays
sequences = np.array(padd_sequences)
labels = np.array(labels)

In [9]:
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Embedding(input_dim = vocab_size, output_dim=8, input_length = seq_length))
model.add(tf.keras.layers.SimpleRNN(32))
model.add(tf.keras.layers.Dense(vocab_size, activation='softmax'))
model.compile(optimizer = 'adam', loss='sparse_categorical_crossentropy', metrics = ['accuracy'])
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
model1 = tf.keras.models.Sequential()
model1.add(tf.keras.layers.Embedding(input_dim = vocab_size, output_dim=8, input_length = seq_length))
model1.add(tf.keras.layers.LSTM(32))
model1.add(tf.keras.layers.Dense(vocab_size, activation='softmax'))
model1.compile(optimizer = 'adam', loss='sparse_categorical_crossentropy', metrics = ['accuracy'])
model1.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
model2 = tf.keras.models.Sequential()
model2.add(tf.keras.layers.Embedding(input_dim = vocab_size, output_dim=8, input_length = seq_length))
model2.add(tf.keras.layers.GRU(32))
model2.add(tf.keras.layers.Dense(vocab_size, activation='softmax'))
model2.compile(optimizer = 'adam', loss='sparse_categorical_crossentropy', metrics = ['accuracy'])
model2.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.fit(sequences, labels, epochs = 10, validation_split = 0.2, batch_size=32)

Epoch 1/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.2667 - loss: 2.5323 - val_accuracy: 0.3768 - val_loss: 1.9685
Epoch 2/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.4122 - loss: 1.8701 - val_accuracy: 0.4391 - val_loss: 1.6981
Epoch 3/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.4614 - loss: 1.6434 - val_accuracy: 0.4929 - val_loss: 1.5310
Epoch 4/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.5046 - loss: 1.4857 - val_accuracy: 0.5315 - val_loss: 1.3965
Epoch 5/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.5467 - loss: 1.3574 - val_accuracy: 0.5576 - val_loss: 1.2959
Epoch 6/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.5718 - loss: 1.2705 - val_accuracy: 0.5717 - val_loss: 1.2320
Epoch 7/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.5811 - loss: 1.2125 - val_accuracy: 0.5822 - val_loss: 1.1788
Epoch 8/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.5910 - loss: 1.1

In [12]:
model1.fit(sequences, labels, epochs = 10, validation_split = 0.2, batch_size=32)

Epoch 1/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.2413 - loss: 2.7052 - val_accuracy: 0.3475 - val_loss: 2.0650
Epoch 2/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.3715 - loss: 1.9680 - val_accuracy: 0.4100 - val_loss: 1.7609
Epoch 3/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.4256 - loss: 1.7039 - val_accuracy: 0.4912 - val_loss: 1.5463
Epoch 4/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.4994 - loss: 1.5119 - val_accuracy: 0.5321 - val_loss: 1.3910
Epoch 5/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.5407 - loss: 1.3714 - val_accuracy: 0.5566 - val_loss: 1.2894
Epoch 6/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.5684 - loss: 1.2623 - val_accuracy: 0.5744 - val_loss: 1.2192
Epoch 7/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.5805 - loss: 1.2027 - val_accuracy: 0.5828 - val_loss: 1.1710
Epoch 8/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.5864 - loss: 1.1635 

In [13]:
model2.fit(sequences, labels, epochs = 10, validation_split = 0.2, batch_size=32)

Epoch 1/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.2779 - loss: 2.5227 - val_accuracy: 0.3893 - val_loss: 1.8929
Epoch 2/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.4061 - loss: 1.8257 - val_accuracy: 0.4482 - val_loss: 1.6408
Epoch 3/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.4743 - loss: 1.5829 - val_accuracy: 0.5353 - val_loss: 1.4033
Epoch 4/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.5449 - loss: 1.3592 - val_accuracy: 0.5651 - val_loss: 1.2688
Epoch 5/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.5720 - loss: 1.2493 - val_accuracy: 0.5783 - val_loss: 1.1980
Epoch 6/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.5828 - loss: 1.1837 - val_accuracy: 0.5849 - val_loss: 1.1545
Epoch 7/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.5923 - loss: 1.1363 - val_accuracy: 0.5834 - val_loss: 1.1283
Epoch 8/10
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.5921 - loss: 1.1192

In [17]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 5, 8)           │           184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         1,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 23)             │           759 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,767 (26.44 KB)

 Trainable params: 2,255 (8.81 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 4,512 (17.63 KB)

In [16]:
model1.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 5, 8)           │           184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │         5,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 23)             │           759 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,575 (72.56 KB)

 Trainable params: 6,191 (24.18 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 12,384 (48.38 KB)

In [18]:
model2.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 5, 8)           │           184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 32)             │         4,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 23)             │           759 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,927 (58.31 KB)

 Trainable params: 4,975 (19.43 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 9,952 (38.88 KB)

In [23]:
def generate_text(seed_text, model_name, next_chars=50):
    result = seed_text
    model = model_name
    for _ in range(next_chars):
        # Convert seed to integer
        seed_encoded = [charidx.get(ch, 0) for ch in seed_text[-seq_length:]]
        seed_padded = pad_sequences([seed_encoded], maxlen=seq_length, padding='post')

        # Predict next character
        prediction = model.predict(seed_padded, verbose=0)
        next_index = np.argmax(prediction)
        next_char = idxchar[next_index]

        result += next_char
        seed_text += next_char

    return result

# Example generation
print(generate_text("Hola", next_chars=100, model_name=model))


Holawh when when when when when when when when when when when when when when when when when when when wh


In [24]:
print(generate_text("Hola", next_chars=100, model_name=model1))

Holawe the one any the one any the one any the one any the one any the one any the one any the one any t


In [25]:
print(generate_text("Hola", next_chars=100, model_name=model2))

Holaad there there there there there there there there there there there there there there there there t
